# 수험번호를 반드시 입력하세요!

**문제를 시작하기 전에, 바로 아래 셀의 `student_id`에 본인의 수험번호를 입력하고 실행하세요.**

예: `student_id = '001234'` — 앞자리 0도 포함하여 따옴표 안에 입력하세요.


In [ ]:
# 본인의 수험번호를 따옴표 안에 입력하세요. 예: '001234'
student_id = ''
if not student_id or not student_id.isascii() or not student_id.isdigit():
    raise ValueError('수험번호를 숫자로만 입력하세요. 앞자리 0도 그대로 적습니다.')
print('수험번호:', student_id)


## 코드 작성 안내
- 답안 작성과 관련 있는 셀은 앞쪽에 `답안 작성 셀`로 시작하는 마크다운 셀이 있습니다.
- 수정이 불가한 셀에는 `# 수정 금지`로 표시가 되어있습니다. 해당 주석이 있는 셀의 코드는 절대 수정하지 마세요.
- 수정 가능한 셀은 `# 수정 가능`으로 표시가 되어있습니다. 다만 셀 안에서 함수명 등 변경하면 안되는 부분도 있으니, 그런 경우 해당 셀 혹은 문항의 안내를 참고하세요.

# 문제 1-2. (예측)

> **[예측 문제]** 공식 샘플의 형식·제약을 그대로 따르고 데이터와 함정만 바꾼 연습 문제입니다. 실제 시험 문제가 아닙니다.

> 우리 쇼핑몰은 상품 사진으로 카테고리를 자동 분류한다. 학습 데이터는 **스튜디오에서 찍은 정돈된 상품컷**(흰 배경, 가운데, 꽉 찬 크기)이다. \
> 이번에 **판매자 앱 업로드 기능**이 열리면서, 판매자가 직접 찍은 사진이 들어오기 시작했다. 이 사진들은 상품이 작게 찍히거나, 한쪽에 치우치거나, 기울어지거나, 어둡거나, 흑백인 경우가 있다.

검증(`val`) 데이터는 **판매자 앱 업로드 사진**에서 뽑았습니다. 학습 데이터는 바꿀 수 없고, **학습용 변환(`train_transform`)만** 수정할 수 있습니다.

## 공통 조건
- 모델은 TinyViT 5M `dist_in22k_ft_in1k`, **Full Fine-Tuning**, 구조·epoch(12)·학습률·batch·Loss는 고정입니다.
- 입력은 **높이 80 × 너비 60 RGB**로 유지합니다(텐서 변환과 정규화도 유지).
- train / val 구분과 정답은 변경하지 마세요.

## 평가 지표
- 최하단 `[Final] Validation macro-F1` 기준: High 0.80 초과 / Mid 0.60 초과 0.80 이하 / Low 0.60 이하

참고 자료는 [오프라인 문서 목록](docs/index.html)에서 볼 수 있습니다. `docs` 폴더를 함께 보관하세요.


## 환경 설정

아래 셀은 가상환경이 설정되어 있으며 커널이 해당 가상환경으로 잡혀 있는지를 체크합니다.
꼭 실행해서 확인 후에 진행하세요. 오류가 발생하면 이 폴더의 README.md를 참고해 환경을 다시 확인하세요.
특히 VS Code, PyCharm 환경에서 Jupyter Notebook 커널을 꼭 확인하세요.


In [ ]:
# 수정 금지
# 먼저 실행해 현재 커널과 GPU를 확인하세요.

import sys
from pathlib import Path

print('Python:', sys.version.split()[0])
print('현재 커널:', sys.executable)
print('작업 폴더:', Path.cwd())

expected_env = Path.cwd() / '.venv'
if Path(sys.prefix).resolve() != expected_env.resolve():
    raise RuntimeError(
        f"{expected_env / 'Scripts' / 'python.exe'}를 노트북 커널로 선택하세요. "
        '환경이 없다면 setup.cmd를 실행하고, 작업 폴더도 노트북이 있는 폴더로 맞추세요.'
    )

try:
    import torch
    import torchvision
    import timm
except ImportError as error:
    raise RuntimeError("필요한 패키지를 불러오지 못했습니다. setup.cmd를 실행한 뒤 커널을 다시 시작하세요.") from error

print('PyTorch:', torch.__version__, '/ torchvision:', torchvision.__version__, '/ timm:', timm.__version__)
print('PyTorch CUDA:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU를 사용할 수 없습니다. 커널, NVIDIA 드라이버, CUDA용 PyTorch 설치를 확인하세요.")

print('GPU:', torch.cuda.get_device_name(0))
torch.zeros(1, device='cuda').sum().item()
print('커널과 GPU 확인 완료. 다음 셀로 진행하세요.')


In [ ]:
# 수정 금지

from pathlib import Path
import random
import json
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import f1_score

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('사용 장치:', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')


In [ ]:
# 수정 금지

data_path = Path('data/f1_2_shift/prepared.npz')
if not data_path.is_file() or not data_path.with_name('classes.json').is_file():
    raise FileNotFoundError('데이터가 없습니다. 먼저 setup.cmd를 실행하고, 작업 폴더를 이 노트북이 있는 폴더로 맞추세요.')

In [ ]:
# 수정 금지

with data_path.with_name('classes.json').open(encoding='utf-8') as file:
    class_names = json.load(file)
num_classes = len(class_names)

with np.load(data_path, allow_pickle=False) as data:
    train_images = data['train_images']
    train_labels = data['train_labels'].reshape(-1)
    val_images = data['val_images']
    val_labels = data['val_labels'].reshape(-1)

print('클래스:', num_classes)
print('학습:', len(train_images), '검증:', len(val_images))
print('이미지 크기:', train_images[0].shape)


In [ ]:
# 수정 가능
# 확인용 그림입니다. 자유롭게 고쳐서 데이터를 살펴보세요.
fig, axes = plt.subplots(2, 8, figsize=(14, 4.5))
for axis, index in zip(axes[0], range(0, len(train_labels), len(train_labels) // 8)):
    axis.imshow(train_images[index]); axis.set_title('train: ' + class_names[train_labels[index]], fontsize=7)
for axis, index in zip(axes[1], range(0, len(val_labels), len(val_labels) // 8)):
    axis.imshow(val_images[index]); axis.set_title('val: ' + class_names[val_labels[index]], fontsize=7)
for axis in axes.flat:
    axis.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 수정 금지

mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean, std),
])


In [ ]:
# 수정 금지

class FashionDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        return self.transform(self.images[index]), int(self.labels[index])


# **답안 작성 셀 - 학습용 변환**

학습 데이터가 검증(판매자 앱) 데이터의 조건에 대응할 수 있도록 `train_transform`을 설계하세요.
- 변환의 추가·삭제·순서·범위·확률을 바꿀 수 있습니다. 이름은 바꾸지 마세요.
- 출력은 정규화된 `torch.float32` 텐서 `(3, 80, 60)`이어야 합니다. `eval_transform`은 수정하지 마세요.
- 제공된 기본 변환 중 **오히려 해가 되는 변환**이 있을 수 있습니다. 선택 이유를 주석으로 남기세요.

### 공식 문서
- [torchvision Transform 전처리 요소 설명](docs/pages/docs.pytorch.org/vision_main_transforms--7c151205d9a0.html)
- [Transform 적용 결과 비교 예제](docs/pages/docs.pytorch.org/vision_0.26_auto_examples_transforms_plot_transforms_illustrations--0d046f058aa3.html)

In [ ]:
# 수정 가능

train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomVerticalFlip(p=0.5),
    T.RandomResizedCrop((80, 60), scale=(0.2, 0.6)),
    T.ColorJitter(hue=0.5),
    T.ToTensor(),
    T.Normalize(mean, std),
])

### Patch Embedding

입력 `(B, 3, 80, 60)`을 특징 맵 `(B, 64, 20, 15)`로 바꿉니다.


In [ ]:
# 수정 금지

class ConvNorm(nn.Sequential):
    """Conv2d와 BatchNorm2d로 구성한 기본 블록."""
    def __init__(self, in_channels, out_channels, kernel_size=1,
                 stride=1, padding=0, groups=1, bn_weight=1.0):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size,
            stride=stride, padding=padding, groups=groups, bias=False,
        )
        self.bn = nn.BatchNorm2d(out_channels)
        nn.init.constant_(self.bn.weight, bn_weight)
        nn.init.zeros_(self.bn.bias)


class PatchEmbed(nn.Module):
    """원본 입력을 특징 맵으로 변환: [B, 3, 80, 60] → [B, 64, 20, 15]."""
    def __init__(self, dim=64):
        super().__init__()
        self.conv1 = ConvNorm(3, dim // 2, 3, stride=2, padding=1)
        self.act = nn.GELU()
        self.conv2 = ConvNorm(dim // 2, dim, 3, stride=2, padding=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.act(x)
        return self.conv2(x)

### 합성곱 블록

첫 stage의 MBConv와 해상도를 줄이는 Patch Merging입니다.


In [ ]:
# 수정 금지

class MBConv(nn.Module):
    """Stage 0: 채널 확장 → depthwise convolution → 채널 축소."""
    def __init__(self, dim, expand_ratio=4):
        super().__init__()
        hidden_dim = dim * expand_ratio
        self.conv1 = ConvNorm(dim, hidden_dim)
        self.act1 = nn.GELU()
        self.conv2 = ConvNorm(hidden_dim, hidden_dim, 3,
                              padding=1, groups=hidden_dim)
        self.act2 = nn.GELU()
        self.conv3 = ConvNorm(hidden_dim, dim, bn_weight=0.0)
        self.act3 = nn.GELU()

    def forward(self, x):
        shortcut = x
        x = self.act1(self.conv1(x))
        x = self.act2(self.conv2(x))
        x = self.conv3(x)
        return self.act3(x + shortcut)


class ConvStage(nn.Module):
    def __init__(self, dim, depth):
        super().__init__()
        self.blocks = nn.Sequential(*[MBConv(dim) for _ in range(depth)])

    def forward(self, x):
        return self.blocks(x)


class PatchMerging(nn.Module):
    """해상도를 절반으로 줄이고 다음 stage의 채널 수로 변환."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.conv1 = ConvNorm(in_dim, out_dim)
        self.act1 = nn.GELU()
        self.conv2 = ConvNorm(out_dim, out_dim, 3, stride=2,
                              padding=1, groups=out_dim)
        self.act2 = nn.GELU()
        self.conv3 = ConvNorm(out_dim, out_dim)

    def forward(self, x):
        x = self.act1(self.conv1(x))
        x = self.act2(self.conv2(x))
        return self.conv3(x)

### Window Attention과 MLP

각 window 안의 위치 관계를 상대 위치 bias로 표현합니다.


In [ ]:
# 수정 금지

class NormMLP(nn.Module):
    def __init__(self, dim, mlp_ratio=4):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, dim * mlp_ratio)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(dim * mlp_ratio, dim)

    def forward(self, x):
        x = self.norm(x)
        x = self.fc1(x)
        x = self.act(x)
        return self.fc2(x)


class WindowAttention(nn.Module):
    """작은 window 안에서 수행하는 multi-head self-attention."""
    def __init__(self, dim, num_heads, window_size):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.norm = nn.LayerNorm(dim)
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

        # window 내 두 위치 사이의 상대 거리에 따른 학습 가능한 bias
        coords = torch.stack(torch.meshgrid(
            torch.arange(window_size), torch.arange(window_size), indexing="ij"
        )).flatten(1)
        offsets = (coords[:, :, None] - coords[:, None, :]).abs()
        bias_indices = offsets[0] * window_size + offsets[1]
        self.register_buffer("attention_bias_idxs", bias_indices, persistent=False)
        self.attention_biases = nn.Parameter(
            torch.zeros(num_heads, window_size * window_size)
        )

    def forward(self, x):
        batch_size, num_tokens, dim = x.shape
        x = self.norm(x)

        # 체크포인트의 QKV 배치 순서: 각 head 안에 Q, K, V가 들어 있습니다.
        qkv = self.qkv(x).reshape(
            batch_size, num_tokens, self.num_heads, 3 * self.head_dim
        )
        qkv = qkv.transpose(1, 2)
        q, k, v = qkv.chunk(3, dim=-1)

        attention = (q * self.scale) @ k.transpose(-2, -1)
        bias = self.attention_biases[:, self.attention_bias_idxs]
        attention = (attention + bias.unsqueeze(0)).softmax(dim=-1)
        x = attention @ v
        x = x.transpose(1, 2).reshape(batch_size, num_tokens, dim)
        return self.proj(x)

### Attention 블록과 Stage

특징 맵을 window로 나누고, 추가한 여백은 attention 후 제거합니다.


In [ ]:
# 수정 금지

class TinyViTBlock(nn.Module):
    """Window attention → local convolution → MLP."""
    def __init__(self, dim, num_heads, window_size):
        super().__init__()
        self.window_size = window_size
        self.attn = WindowAttention(dim, num_heads, window_size)
        self.local_conv = ConvNorm(dim, dim, 3, padding=1, groups=dim)
        self.mlp = NormMLP(dim)

    def forward(self, x):
        # 이 블록 안에서는 [B, H, W, C] 순서로 처리합니다.
        batch_size, height, width, channels = x.shape
        shortcut = x
        window = self.window_size

        # 해상도가 window 크기의 배수가 아닐 때 오른쪽·아래쪽에 padding
        pad_h = (window - height % window) % window
        pad_w = (window - width % window) % window
        x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        padded_h, padded_w = height + pad_h, width + pad_w
        num_h, num_w = padded_h // window, padded_w // window

        # 이미지 → 여러 window
        x = x.reshape(batch_size, num_h, window, num_w, window, channels)
        x = x.transpose(2, 3).reshape(-1, window * window, channels)
        x = self.attn(x)

        # 여러 window → 이미지
        x = x.reshape(batch_size, num_h, num_w, window, window, channels)
        x = x.transpose(2, 3).reshape(batch_size, padded_h, padded_w, channels)
        x = x[:, :height, :width, :].contiguous()
        x = shortcut + x

        x = x.permute(0, 3, 1, 2)
        x = self.local_conv(x)
        x = x.flatten(2).transpose(1, 2)
        x = x + self.mlp(x)
        return x.reshape(batch_size, height, width, channels)


class TinyViTStage(nn.Module):
    def __init__(self, in_dim, out_dim, depth, num_heads, window_size):
        super().__init__()
        self.downsample = PatchMerging(in_dim, out_dim)
        self.blocks = nn.Sequential(*[
            TinyViTBlock(out_dim, num_heads, window_size) for _ in range(depth)
        ])

    def forward(self, x):
        x = self.downsample(x)
        x = x.permute(0, 2, 3, 1)
        x = self.blocks(x)
        return x.permute(0, 3, 1, 2)

### 사전학습 가중치

1,000종 분류 가중치를 `strict=True`로 불러온 뒤 분류기를 20종으로 교체합니다. 상대 위치 bias만 작은 window에 맞춰 보간합니다. 이 셀은 수정하지 마세요.


In [ ]:
# 수정 금지
# 사전학습 가중치 로딩과 출력 분류기입니다.

from safetensors.torch import load_file


class ClassifierHead(nn.Module):
    def __init__(self, dim, num_classes):
        super().__init__()
        self.norm = nn.LayerNorm(dim, eps=1e-5)
        self.fc = nn.Linear(dim, num_classes)

    def forward(self, x):
        x = x.mean(dim=(2, 3))
        x = self.norm(x)
        return self.fc(x)


def load_pretrained_weights(model):
    """제공된 로컬 가중치를 읽고 상대 위치 bias만 입력 크기에 맞게 보간합니다."""
    path = Path('tiny_vit_5m_224.dist_in22k_ft_in1k.safetensors')
    if not path.is_file():
        raise FileNotFoundError(
            f'사전학습 가중치 파일을 찾을 수 없습니다: {path.resolve()}\n'
            '제공된 가중치 파일을 노트북과 같은 폴더에 두고, 작업 폴더를 해당 폴더로 맞추세요.'
        )
    weights = load_file(str(path))
    target = model.state_dict()
    for name, value in weights.items():
        if name.endswith('attention_biases') and value.shape != target[name].shape:
            num_heads, length = value.shape
            old_window = int(length ** 0.5)
            new_window = int(target[name].shape[1] ** 0.5)
            weights[name] = F.interpolate(
                value.float().reshape(1, num_heads, old_window, old_window),
                size=(new_window, new_window),
                mode='bicubic', antialias=True,
            ).reshape(num_heads, new_window * new_window)
    model.load_state_dict(weights, strict=True)


## 고정 모델 (Full Fine-Tuning)
아래 모델·Loss·학습 설정은 수정하지 마세요.

In [ ]:
# 수정 금지

class FashionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = PatchEmbed(dim=64)
        self.stages = nn.Sequential(
            ConvStage(dim=64, depth=2),
            TinyViTStage(64, 128, depth=2, num_heads=4, window_size=3),
            TinyViTStage(128, 160, depth=6, num_heads=5, window_size=5),
            TinyViTStage(160, 320, depth=2, num_heads=10, window_size=3),
        )
        self.head = ClassifierHead(320, num_classes=1000)
        load_pretrained_weights(self)
        self.head.fc = nn.Linear(320, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        for stage in self.stages:
            x = stage(x)
        x = x.mean(dim=(2, 3))
        x = self.head.norm(x)
        return self.head.fc(x)

In [ ]:
# 수정 금지

batch_size = 64
learning_rate = 0.0001
epochs = 12
loss_fn = nn.CrossEntropyLoss()
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)
checkpoint = output_dir / f'F1_2_{student_id}.pt'

In [ ]:
# 수정 금지

def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0
    sample_count = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item() * len(labels)
        sample_count += len(labels)
    return total_loss / sample_count


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    predictions, targets = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            total_loss += loss_fn(logits, labels).item() * len(labels)
            predictions.extend(logits.argmax(1).cpu().tolist())
            targets.extend(labels.cpu().tolist())
    score = f1_score(
        targets, predictions, labels=list(range(num_classes)), average='macro', zero_division=0
    )
    return total_loss / len(targets), score


In [ ]:
# 수정 금지

train_dataset = FashionDataset(train_images, train_labels, train_transform)
val_dataset = FashionDataset(val_images, val_labels, eval_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

In [ ]:
# 수정 금지

torch.manual_seed(42)
model = FashionModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
best_f1 = -1
history = []
started = time.perf_counter()

for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
    val_loss, val_f1 = evaluate(model, val_loader, loss_fn)
    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_f1': val_f1,
    })
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), checkpoint)
    print(f'{epoch + 1:2d}/{epochs}  학습 Loss: {train_loss:.4f}  검증 F1: {val_f1:.4f}')

print(f'학습 시간: {time.perf_counter() - started:.1f}초')

In [ ]:
# 수정 금지

plt.plot([row['epoch'] for row in history], [row['val_f1'] for row in history])
plt.xlabel('Epoch')
plt.ylabel('Validation macro-F1')
plt.show()


In [ ]:
# 수정 금지

model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
val_loss, val_f1 = evaluate(model, val_loader, loss_fn)
print(f'[Final] Validation macro-F1: {val_f1:.4f}')

## 제출
- 완성한 노트북 `.ipynb`
- Best 체크포인트 `outputs/F1_2_<수험번호>.pt`
- 실제로 실행한 validation 결과를 노트북에 남기세요.